# Phase 4: ImageNet transfer learning
Compare a trained classifier head and partial fine-tuning using the existing RealWaste split. Use MobileNetV3 first, then enable the remaining architectures in MODEL_NAMES.

In [ ]:
from pathlib import Path, PurePosixPath
import json
import random
import sys
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models

MODEL_NAMES = ['mobilenet_v3_large']  # Add efficientnet_b0 and googlenet after the first run.
HEAD_EPOCHS = 6
FINETUNE_EPOCHS = 10
BATCH_SIZE = 32

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    project_root = Path('/content/drive/MyDrive/IML_RealWaste')
else:
    project_root = Path.cwd()
    if project_root.name == 'notebooks':
        project_root = project_root.parent

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, torch.cuda.get_device_name() if device.type == 'cuda' else '')

## Load the shared manifests

In [ ]:
data_dir = project_root / 'data'
metadata = json.loads((data_dir / 'dataset_meta.json').read_text(encoding='utf-8'))
classes = metadata['classes']
label_ids = list(range(len(classes)))
seed = metadata['seed']
frames = {split: pd.read_csv(data_dir / f'{split}.csv') for split in ('train', 'val', 'test')}

# Colab reads images from its local disk to avoid repeated Drive filesystem access.
# Keep manifests and experiment outputs in Drive.
if 'google.colab' in sys.modules:
    from urllib.request import urlretrieve
    from zipfile import ZipFile
    import shutil

    cache_root = Path('/content/iml_realwaste_cache')
    dataset_root = cache_root / 'RealWaste'
    required_paths = {
        PurePosixPath(row.label) / PurePosixPath(row.filepath.replace(chr(92), '/')).name
        for frame in frames.values() for row in frame.itertuples()
    }
    missing_paths = [path for path in required_paths if not (dataset_root / str(path)).is_file()]
    if missing_paths:
        cache_root.mkdir(parents=True, exist_ok=True)
        archive_path = cache_root / 'realwaste.zip'
        download_path = cache_root / 'realwaste.zip.part'
        if not archive_path.is_file():
            print('Downloading official RealWaste dataset to runtime storage...', flush=True)
            urlretrieve('https://archive.ics.uci.edu/static/public/908/realwaste.zip', download_path)
            download_path.replace(archive_path)
        with ZipFile(archive_path) as archive:
            members = {
                PurePosixPath(*PurePosixPath(name).parts[-2:]): name
                for name in archive.namelist() if not name.endswith('/')
            }
            absent = set(missing_paths) - members.keys()
            if absent:
                raise FileNotFoundError(f'Official archive lacks manifest images: {sorted(map(str, absent))[:5]}')
            for relative_path in missing_paths:
                destination = dataset_root / str(relative_path)
                destination.parent.mkdir(parents=True, exist_ok=True)
                temporary = destination.with_suffix('.part')
                with archive.open(members[relative_path]) as source, temporary.open('wb') as target:
                    shutil.copyfileobj(source, target)
                temporary.replace(destination)
else:
    dataset_roots = [
        data_dir / 'realwaste-main' / 'RealWaste',
        project_root / 'realwaste-main' / 'RealWaste',
    ]
    dataset_root = next((root for root in dataset_roots if all((root / name).is_dir() for name in classes)), None)
    if dataset_root is None:
        raise FileNotFoundError(f'RealWaste class folders not found under {dataset_roots}')
print(f'Image source: {dataset_root}')

seen = set()
for split, frame in frames.items():
    if len(frame) != metadata[f'n_{split}'] or not frame['label'].map(metadata['label2idx']).eq(frame['label_idx']).all():
        raise ValueError(f'{split}: split size or labels do not match metadata')
    frame['relative_path'] = [
        str(PurePosixPath(label) / PurePosixPath(path.replace(chr(92), '/')).name)
        for label, path in zip(frame['label'], frame['filepath'])
    ]
    paths = set(frame['relative_path'])
    if len(paths) != len(frame) or paths & seen:
        raise ValueError(f'{split}: duplicate image paths within or across splits')
    seen.update(paths)
    missing = [path for path in paths if not (dataset_root / path).is_file()]
    if missing:
        raise FileNotFoundError(f'{split}: missing {len(missing)} images, e.g. {missing[0]}')
    for path in paths:
        with Image.open(dataset_root / path) as image:
            image.verify()
    print(split, len(frame), 'images verified')

## Build datasets and initialize models
Both training stages use the same preprocessing so the stage comparison isolates the change in training strategy.

In [ ]:
class WasteImages(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame['relative_path'].tolist()
        self.targets = frame['label_idx'].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        with Image.open(dataset_root / self.paths[index]) as image:
            return self.transform(image.convert('RGB')), self.targets[index]

def build_model(name):
    if name == 'mobilenet_v3_large':
        weights = models.MobileNet_V3_Large_Weights.DEFAULT
        model = models.mobilenet_v3_large(weights=weights)
        head = model.classifier
        head[-1] = nn.Linear(head[-1].in_features, len(classes))
        blocks = [model.features[-2], model.features[-1]]
    elif name == 'efficientnet_b0':
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        head = model.classifier
        head[-1] = nn.Linear(head[-1].in_features, len(classes))
        blocks = [model.features[-2], model.features[-1]]
    elif name == 'googlenet':
        weights = models.GoogLeNet_Weights.DEFAULT
        model = models.googlenet(weights=weights, aux_logits=False)
        model.fc = nn.Linear(model.fc.in_features, len(classes))
        head = model.fc
        blocks = [model.inception5a, model.inception5b]
    else:
        raise ValueError(f'Unknown model: {name}')
    return model.to(device), head, blocks, weights

## Train and select on validation macro F1

In [ ]:
def run_epoch(model, loader, optimizer=None, trainable_modules=()):
    # Keep the frozen backbone, including its BatchNorm buffers, in eval mode.
    model.eval()
    for module in trainable_modules:
        module.train()
    loss_total = 0.0
    expected, predicted = [], []
    with torch.set_grad_enabled(optimizer is not None):
        for images, labels in loader:
            labels = labels.to(device)
            logits = model(images.to(device))
            loss = nn.functional.cross_entropy(logits, labels)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            loss_total += loss.item() * len(labels)
            expected.extend(labels.cpu().tolist())
            predicted.extend(logits.argmax(dim=1).cpu().tolist())
    return {
        'loss': loss_total / len(expected),
        'accuracy': accuracy_score(expected, predicted),
        'macro_f1': f1_score(expected, predicted, labels=label_ids, average='macro', zero_division=0),
    }, expected, predicted

def train_stage(model, head, blocks, loaders, name, stage, epochs, history):
    model.requires_grad_(False)
    head.requires_grad_(True)
    parameter_groups = [{'params': head.parameters(), 'lr': 1e-3 if not blocks else 3e-4}]
    for block in blocks:
        block.requires_grad_(True)
        parameter_groups.append({'params': block.parameters(), 'lr': 3e-5})
    optimizer = torch.optim.AdamW(parameter_groups, weight_decay=1e-4)
    path = models_dir / f'phase4_{name}_{stage}.pth'
    best_score = -1
    best_epoch = 0
    for epoch in range(1, epochs + 1):
        train, _, _ = run_epoch(model, loaders['train'], optimizer, (head, *blocks))
        val, _, _ = run_epoch(model, loaders['val'])
        history.append({'stage': stage, 'epoch': epoch, 'train_loss': train['loss'],
                        'val_loss': val['loss'], 'val_accuracy': val['accuracy'], 'val_macro_f1': val['macro_f1']})
        if val['macro_f1'] > best_score:
            best_score, best_epoch = val['macro_f1'], epoch
            temp_path = path.with_suffix('.tmp')
            torch.save(model.state_dict(), temp_path)
            temp_path.replace(path)
        pd.DataFrame(history).to_csv(metrics_dir / f'phase4_{name}_history.csv', index=False)
        print(f"{name} {stage} {epoch}/{epochs} | val F1 {val['macro_f1']:.4f}", flush=True)
    return path, best_score, best_epoch

## Run the experiments and export test results

In [ ]:
if 'google.colab' in sys.modules and device.type != 'cuda':
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then rerun all cells.')
    
results_dir = project_root / 'results'
metrics_dir, models_dir, plots_dir = [results_dir / name for name in ('metrics', 'models', 'plots')]
for directory in (metrics_dir, models_dir, plots_dir):
    directory.mkdir(parents=True, exist_ok=True)

summary = []
for name in MODEL_NAMES:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    model, head, blocks, weights = build_model(name)
    loaders = {
        split: DataLoader(WasteImages(frame, weights.transforms()), batch_size=BATCH_SIZE,
                          shuffle=split == 'train', pin_memory=device.type == 'cuda',
                          generator=torch.Generator().manual_seed(seed))
        for split, frame in frames.items()
    }
    images, labels = next(iter(loaders['val']))
    model.eval()
    with torch.inference_mode():
        logits = model(images.to(device))
    if logits.shape != (len(labels), len(classes)):
        raise ValueError(f'{name}: unexpected output shape {logits.shape}')

    history = []
    start = perf_counter()
    frozen = train_stage(model, head, [], loaders, name, 'head', HEAD_EPOCHS, history)
    model.load_state_dict(torch.load(frozen[0], map_location=device, weights_only=True))
    tuned = train_stage(model, head, blocks, loaders, name, 'finetune', FINETUNE_EPOCHS, history)
    train_time = perf_counter() - start
    stage, (checkpoint, val_f1, epoch) = ('finetune', tuned) if tuned[1] > frozen[1] else ('head', frozen)

    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    score, expected, predicted = run_epoch(model, loaders['test'])
    report = classification_report(expected, predicted, labels=label_ids, target_names=classes,
                                   output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(metrics_dir / f'phase4_{name}_report.csv')
    pd.DataFrame({
        'relative_path': frames['test']['relative_path'],
        'true_label': [classes[index] for index in expected],
        'predicted_label': [classes[index] for index in predicted],
    }).to_csv(metrics_dir / f'phase4_{name}_predictions.csv', index=False)

    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay.from_predictions(expected, predicted, labels=label_ids,
                                            display_labels=classes, xticks_rotation=45, ax=ax)
    ax.set_title(f'{name} test confusion matrix')
    fig.savefig(plots_dir / f'phase4_{name}_confusion.png', bbox_inches='tight')
    plt.close(fig)
    result = {'model': name, 'weights': weights.name, 'selected_stage': stage, 'selected_epoch': epoch,
              'val_macro_f1': val_f1, 'test_accuracy': score['accuracy'], 'test_macro_f1': score['macro_f1'],
              'test_loss': score['loss'], 'train_seconds': train_time}
    pd.DataFrame([result]).to_csv(metrics_dir / f'phase4_{name}_summary.csv', index=False)
    summary.append(result)
pd.DataFrame(summary)